# 레슨 1-1. 신용평가모형은 무엇인가

**예상 시간** 40분 · **선행 지식** 없음 · **이론 페이지** `docs/theory/stage_01.html`, `docs/theory/stage_15.html`

> 이 노트북은 *교육용*입니다. 여기서 만드는 숫자는 실제 은행의 규제 승인이나 실제 신용위험과 무관합니다.

## 학습 목표
1. 신용평가모형이 *무엇을 입력받아 무엇을 출력하는지* 한 문장으로 설명한다.
2. **모형(예측)** 과 **정책(컷오프)** 이 다른 것임을 구분한다.
3. 부도율을 직접 계산하고, 표본이 작을 때 숫자를 믿으면 안 되는 이유를 *숫자로* 확인한다.

In [ ]:
# [환경 준비] 맨 위 셀부터 순서대로 실행하세요. (Colab/로컬 모두 동작)
import os
import subprocess
import sys
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/nh15312323-art/CreditModel.git"


def find_or_fetch_project() -> Path:
    for base in (Path.cwd(), Path.cwd().parent, Path("/content/credit-model-mastery-lab")):
        if (base / "src" / "creditlab").exists():
            return base
    if not IN_COLAB:
        raise RuntimeError("프로젝트 폴더를 찾지 못했습니다. 저장소 루트(또는 notebooks/)에서 노트북을 여세요.")
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/credit-model-mastery-lab"], check=True)
    else:
        from google.colab import files  # Colab 전용

        print("credit-model-mastery-lab.zip 파일을 선택해 업로드하세요.")
        files.upload()
        subprocess.run(["unzip", "-q", "-o", "credit-model-mastery-lab.zip", "-d", "/content"], check=True)
    return Path("/content/credit-model-mastery-lab")


PROJECT_ROOT = find_or_fetch_project()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
if IN_COLAB:  # UCI 원본이 엑셀(.xls)이라 읽기 엔진이 필요합니다
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xlrd", "openpyxl"], check=False)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import creditlab
from creditlab import environment_info, set_seed

SEED = set_seed(42)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
print("프로젝트 위치:", PROJECT_ROOT)
print("creditlab", creditlab.__version__, "| 시드", SEED)
print({k: v for k, v in environment_info(SEED)["packages"].items()})

## 1. 개념 — 신용평가모형이란?

**한 문장 정의**: 대출 *신청 시점*에 알 수 있는 정보로, 정해진 기간(성과관찰기간) 안에 *부도가 날 가능성*을 숫자(확률·점수)로 바꿔 주는 도구.

| 구분 | 질문 | 예 |
|---|---|---|
| 순위변별력 | 누가 *더* 위험한가? | A가 B보다 위험하다 |
| 확률정확성 | 실제로 몇 %가 부도나는가? | 이 점수대는 약 3% |

둘은 서로 다른 능력입니다. 순위만 잘 맞춰도 확률 값은 틀릴 수 있습니다(레슨 9단계에서 다시 만납니다).

### 모형 ≠ 정책
- **모형**: 점수/확률을 *만든다*.
- **정책(컷오프, 한도, 금리)**: 손실·수익·위험선호를 반영해 *사람이 정한다*.
같은 모형도 정책이 다르면 승인율과 손실이 달라집니다. 그래서 둘을 따로 평가합니다.

### 모형 개발의 큰 흐름
`문제·타깃 정의 → 데이터 품질 점검 → 변수 분석 → 모형 학습 → 독립 검증 → 점수·등급화 → 정책 → 운영 모니터링`

이 프로그램은 이 흐름 전체를 단계별로 직접 해 봅니다.

## 2. 아주 작은 예제 — 10명의 신청자

숫자를 손으로 확인할 수 있도록 일부러 작고 과장된 표를 씁니다(실제 부도율은 이렇게 높지 않습니다).

In [ ]:
toy = pd.DataFrame(
    {
        "applicant": list("ABCDEFGHIJ"),
        "num_delinq_12m": [0, 0, 0, 1, 0, 2, 1, 3, 0, 2],  # 최근 12개월 연체 건수
        "default": [0, 0, 0, 1, 1, 1, 0, 1, 0, 0],  # 1 = 부도, 0 = 정상
    }
)
toy

### ✋ 먼저 예측해 보세요
연체 건수가 **2건**인 그룹(F, J)의 부도율은 몇 %일까요? 아래 `prediction_pct` 에 숫자(예: 30)를 적고 실행하세요. 비워 두어도 실행은 됩니다.

In [ ]:
prediction_pct = None  # ← 여기에 예측(%)을 적어 보세요

by_delinq = toy.groupby("num_delinq_12m")["default"].agg(n="size", defaults="sum", default_rate="mean")
by_delinq["default_rate"] = by_delinq["default_rate"] * 100  # 퍼센트
actual_pct = by_delinq.loc[2, "default_rate"]
if prediction_pct is None:
    print("예측을 적지 않았습니다. 그래도 결과를 봅시다 →", f"연체 2건 그룹의 부도율 = {actual_pct:.0f}%")
else:
    print(f"예측 {prediction_pct}% / 실제 {actual_pct:.0f}% / 차이 {abs(prediction_pct - actual_pct):.0f}%p")
by_delinq

### 숫자를 믿어도 될까? — 표본 크기와 신뢰구간
연체 3건 그룹은 **1명**뿐이고 그 1명이 부도라 부도율이 100%입니다. 이 그룹은 정말 100% 부도날까요?
부도율의 **95% 신뢰구간(Wilson)** 으로 확인해 봅니다. 구간이 넓을수록 숫자를 덜 믿어야 합니다.

In [ ]:
from creditlab.data import default_rate_ci

for k, g in toy.groupby("num_delinq_12m"):
    low, high = default_rate_ci(g["default"])
    print(f"연체 {k}건: n={len(g)}, 부도율 {g['default'].mean():.0%}, 95% 구간 {low:.0%} ~ {high:.0%}")

**관찰**: 건수가 1~2인 그룹은 구간이 0%에서 100%에 가까울 만큼 넓습니다. 그래서 *등급을 나눌 때 한 구간에 최소 몇 건·몇 개의 부도가 있어야 하는지* 를 따로 정합니다(5단계 비닝, 7단계 등급화).

## 3. 가장 단순한 '모형' 만들어 보기
"그룹별 부도율을 그 그룹 모두의 예측확률로 쓴다"가 가장 단순한 모형입니다. 비교 기준(baseline)은 "모두에게 전체 평균 부도율을 준다"입니다.

In [ ]:
toy["pred_group"] = toy.groupby("num_delinq_12m")["default"].transform("mean")  # 그룹 모형
toy["pred_base"] = toy["default"].mean()  # 기준선: 모두 같은 값
toy[["applicant", "num_delinq_12m", "default", "pred_group", "pred_base"]]

**주의 ⚠️** 이 표는 *만든 데이터로 그대로 평가*한 것이라 좋아 보일 수밖에 없습니다. 모형은 **처음 보는 데이터**에서 평가해야 하며, 이 원칙(분할·누수 방지)이 8단계의 핵심입니다.

## 4. 실습 — 개발 흐름 순서 맞추기
아래 5개 단계를 올바른 순서로 나열하세요. `my_order` 에 번호를 순서대로 적습니다(예: `[3, 1, 5, 2, 4]`).

In [ ]:
steps = {
    1: "독립 검증(처음 보는 데이터에서 성능 확인)",
    2: "운영·모니터링",
    3: "모형 학습",
    4: "문제·타깃 정의",
    5: "데이터 품질 점검",
}
for k, v in steps.items():
    print(k, v)

my_order = None  # ← 예: [4, 5, 3, 1, 2]
correct_order = [4, 5, 3, 1, 2]
if my_order is None:
    print("\nmy_order 를 적고 다시 실행하세요.")
elif my_order == correct_order:
    print("\n정답입니다. 정의 → 품질 점검 → 학습 → 검증 → 운영 순서입니다.")
else:
    print("\n순서가 다릅니다. 힌트: '무엇을 맞힐지(타깃)'가 정해지기 전에는 데이터를 평가할 수 없습니다.")

## 5. 퀴즈
`answers` 에 `"A"`~`"D"` 를 적고 실행하세요. 막히면 `hint(q, 1)` → `hint(q, 2)` 순서로 힌트를 여세요.

In [ ]:
from creditlab.learning import Question, check, grade, hint, show

QUIZ = [
    Question("Q1", "모형과 정책", "컷오프(승인/거절 기준 점수)에 대한 설명으로 가장 적절한 것은?",
             {"A": "모형이 자동으로 최적 컷오프를 정해 준다",
              "B": "모형은 점수·확률을 만들고, 컷오프는 손실·수익·위험선호를 반영해 따로 정하는 정책 결정이다",
              "C": "컷오프는 AUROC 값으로 결정된다",
              "D": "컷오프는 규제가 하나로 정해 준다"},
             "B", "모형은 위험을 숫자로 만들고, 컷오프·한도·금리는 사업 정책입니다. 그래서 예측모형과 정책을 분리해 평가합니다.",
             ("모형이 하는 일과 사람이 정하는 일을 나눠 보세요.", "컷오프를 바꾸면 모형 자체가 바뀌나요?")),
    Question("Q2", "부도율", "100건 중 8건이 부도라면 부도율은?",
             {"A": "8%", "B": "0.8%", "C": "92%", "D": "12.5%"},
             "A", "부도율 = 부도 건수 / 전체 건수 = 8/100 = 8% 입니다.", ("분모는 전체 건수입니다.",)),
    Question("Q3", "표본 크기", "단 1건만 있는 그룹의 부도율이 100%입니다. '이 그룹은 반드시 부도난다'고 결론 내려도 될까요?",
             {"A": "된다. 100%면 확실하다", "B": "안 된다. 표본이 너무 작아 신뢰구간이 매우 넓다",
              "C": "건수가 1이면 오히려 정확하다", "D": "부도율이 100%면 항상 사실이다"},
             "B", "n=1 의 Wilson 95% 구간은 매우 넓습니다(위 실습 참고). 소표본 비율은 불안정합니다.",
             ("위 실습에서 연체 3건 그룹의 구간을 다시 보세요.",)),
    Question("Q4", "시점·누수", "대출 신청 시점 모형에 '대출 실행 이후 6개월간의 연체 횟수'를 설명변수로 넣으면?",
             {"A": "성능이 좋아지므로 권장된다", "B": "신청 시점에는 알 수 없는 미래 정보라서 누수(leakage)다",
              "C": "데이터가 많아져 안정적이다", "D": "상관이 높으니 우선 선택한다"},
             "B", "모형은 *사용하는 시점에 알 수 있는 정보*만 써야 합니다. 미래 정보는 성능을 부풀리지만 실제로는 쓸 수 없습니다.",
             ("모형을 실제로 쓰는 시점에 그 값이 존재하나요?",)),
    Question("Q5", "교육 데이터의 한계", "공개 교육 데이터에서 변별력 지표(AUROC)가 매우 높게 나왔습니다. 올바른 해석은?",
             {"A": "실제 은행에서도 같은 성능이 나온다", "B": "규제 승인이 가능하다",
              "C": "그 데이터·기간·부도 정의에서의 결과일 뿐 실제 운영 성능을 보장하지 않는다", "D": "완벽한 모형이다"},
             "C", "데이터의 시대·국가·상품·부도 정의가 다르면 성능도 달라집니다. 교육용 결과를 실제로 일반화하지 않습니다.",
             ("이 데이터가 내가 일하는 시장·시점·상품을 대표하나요?",)),
]

answers = {"Q1": None, "Q2": None, "Q3": None, "Q4": None, "Q5": None}  # ← "A"~"D" 를 적으세요

for q in QUIZ:
    show(q)
    print()

In [ ]:
for q in QUIZ:
    check(q, answers[q.qid])

if all(v is not None for v in answers.values()):
    result = grade(QUIZ, answers)
    print("\n점수:", result["score"], "/", result["total"], "| 통과(75% 기준):", result["passed"])
    if result["concepts_to_review"]:
        print("다시 볼 개념:", ", ".join(result["concepts_to_review"]))

## 6. 과제와 정리

**과제 (제출물: 짧은 글 5줄 이내)**
1. 내 말로 신용평가모형을 한 문장으로 정의해 보세요.
2. 위 10명 예제에서 부도율을 *믿기 어려운* 그룹은 어디이고 이유는 무엇인가요?
3. "모형이 좋으면 컷오프는 자동으로 정해진다"는 주장에 반박해 보세요.

**체크포인트**: 퀴즈 4/5 이상 · 위 3개 과제를 자기 말로 작성

**참고자료**
- 이론 설명 페이지: `docs/theory/stage_01.html`(문제 정의), `docs/theory/stage_15.html`(운영)
- Basel Committee on Banking Supervision, *International Convergence of Capital Measurement and Capital Standards: A Revised Framework* (2006) — 부도 정의의 국제 기준 맥락 (링크: https://www.bis.org/bcbs/)
- 확인일 2026-10-18. 규정은 개정될 수 있으므로 실제 적용 시 최신 원문을 확인하세요.

**다음 레슨**: 1-2. 행·열·타깃 이해하기